# User Behavior Anomaly Detection for Cybersecurity - 2
### Isolation Forest for Login Behavior Analysis - Login Risk and Account Takeover

**Author:** MiyukiMol  
**Target roles:** Data Scientist · Security Analytics · Threat Detection  
**Tools:** Python, pandas, NumPy, scikit-learn, Matplotlib, Seaborn, Google Colab  
**Dataset:** [Login Data Set for Risk-Based Authentication (RBA)](https://www.kaggle.com/datasets/dasgroup/rba-dataset)


---



## 1. Project Overview

### 1.1 Business Context

In cybersecurity, detecting suspicious login activity is challenging because confirmed attack labels are often limited or unavailable.

Traditional supervised machine learning requires labelled examples to learn the difference between legitimate and malicious activity.

Unsupervised anomaly detection offers an alternative approach by identifying unusual patterns without requiring attack labels during model training.

### 1.2 Project Objective

The objective of this notebook is to implement an Isolation Forest model to identify potentially anomalous login events.

The model will learn patterns in login behavior and assign anomaly scores to individual events.

### 1.3 Research Questions

- Can Isolation Forest identify unusual login behavior?
- Which login characteristics are associated with detected anomalies?
- How do detected anomalies compare with known account-takeover labels?
- What are the limitations of unsupervised anomaly detection in cybersecurity?

### 1.4 Notebook Structure

This notebook follows a structured unsupervised anomaly detection workflow:

1. **Project Overview** — Project objectives and business context
2. **Environment Setup** — Importing libraries and configuring the environment
3. **Dataset Loading** — Loading the dataset
4. **Data Preparation** — Data inspection and preparation
5. **Feature Engineering** — Creating features for anomaly detection
6. **Data Preprocessing** — Handling missing values and encoding categorical variables
7. **Train Isolation Forest** — Training the unsupervised anomaly detection model
8. **Anomaly Detection** — Generating anomaly scores and identifying unusual login events
9. **Model Evaluation** — Comparing detected anomalies with account-takeover labels
10. **Feature Analysis** — Analyzing the characteristics of detected anomalies
11. **Key Findings** — Summarizing results and limitations
12. **Conclusion and Next Steps** — Discussing improvements and future work

### 1.5 Important Considerations

The `Is Account Takeover` column will be excluded from model training and used only for post-training evaluation.

An anomaly does not necessarily represent a cyberattack. It indicates behavior that differs from the patterns learned by the model.

## 2 - Environment Setup

### 2.1 Import Libraries

In [21]:
# 2.1 Import Libraries

# Data manipulation
import pandas as pd
import numpy as np
import os
import zipfile
from getpass import getpass

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning and preprocessing
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Model evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

# Utilities
import warnings
import random

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries imported successfully.")
print("Random state:", RANDOM_STATE)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Libraries imported successfully.
Random state: 42
Pandas version: 2.2.3
NumPy version: 2.1.3


### 2.2 Environment Configuration

A fixed random state is used to improve reproducibility across experiments.

The visualization settings are configured to make charts easier to interpret.

The following configuration will be used throughout this notebook:

- **Random state:** 42
- **Visualization style:** Seaborn whitegrid
- **Machine learning library:** Scikit-learn

**特に重要な設計方針**
今回のデータセットには、ATO陽性ラベルが141件しかありません。

そのため、次の方針で実装します。
1. 教師なし学習： Is Account Takeover をIsolation Forestの入力から除外する。
2. 特徴量： ログイン時間、ログイン成功・失敗、国、デバイス、ユーザーのログイン頻度などを検討する。
3. 評価： 学習に使用していないATOラベルと検出結果を比較する。
4. データ分割： 時系列によるデータリークを避ける。
5. 不均衡対策： Accuracyだけでなく、Precision、Recall、F1、PR-AUCを確認する。

重要： 10,000行のEDAサンプルだけではATO陽性例がないため、Notebook 02では元データからATO陽性例を含むデータを取得する必要があります。また、Isolation Forestの異常検知結果は攻撃の確定判定ではありません。

## 3 - Dataset Acquisition and Loading

This project uses the Login Data Set for Risk-Based Authentication (RBA) from Kaggle.

The dataset contains approximately 31.27 million login events, including login timestamps, user identifiers, login outcomes, device information, geographical information, and account-takeover labels.

A major challenge is the extreme class imbalance: only 141 events are labelled as account takeovers.

Therefore, this notebook uses a memory-efficient data-loading strategy to preserve rare positive cases while preparing data for unsupervised anomaly detection.

**Dataset source:** [Login Data Set for Risk-Based Authentication (RBA)](https://www.kaggle.com/datasets/dasgroup/rba-dataset)

### 3.1 Install Kaggle CLI

we will download the "**Login Data Set for Risk-Based Authentication**" dataset from Kaggle and load a manageable sample without extracting the entire large CSV to disk.

https://www.kaggle.com/datasets/dasgroup/rba-dataset

In [22]:
# Install the Kaggle CLI in the current Colab runtime.
%pip -q install kaggle

### 3.2 Configure Kaggle API Credentials

In [23]:
# Enter the Kaggle API token without displaying it.
# For a public portfolio, consider using Google Colab Secrets instead.
kaggle_token = getpass("Paste your Kaggle API token: ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token

Paste your Kaggle API token: ··········


### 3.3 Download the Dataset

In [24]:
# Download the dataset ZIP.
!kaggle datasets download -d dasgroup/rba-dataset -p /content

Dataset URL: https://www.kaggle.com/datasets/dasgroup/rba-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
rba-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


### 3.4 Verify the Downloaded ZIP Archive

In [25]:
ZIP_PATH = "/content/rba-dataset.zip"

# Check whether the ZIP file exists
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "Dataset ZIP file was not found. Please check the download step."
    )

# Verify that the file is a valid ZIP archive
if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError("The downloaded file is not a valid ZIP archive.")

file_size_gb = os.path.getsize(ZIP_PATH) / (1024 ** 3)

print("ZIP archive verified successfully.")
print(f"File path: {ZIP_PATH}")
print(f"File size: {file_size_gb:.2f} GB")

ZIP archive verified successfully.
File path: /content/rba-dataset.zip
File size: 1.10 GB


### 3.5 Identify the CSV File

In [26]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    csv_files = [
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]

    if not csv_files:
        raise FileNotFoundError(
            "No CSV file was found in the ZIP archive."
        )

    print("CSV file(s) found:")

    for csv_file in csv_files:
        print(f"- {csv_file}")

if len(csv_files) > 1:
    raise ValueError(
        "Multiple CSV files found. Please identify the correct dataset."
    )

CSV_NAME = csv_files[0]

print(f"\nSelected CSV file: {CSV_NAME}")

CSV file(s) found:
- rba-dataset.csv

Selected CSV file: rba-dataset.csv


### 3.6 Inspect the Dataset Structure

In [27]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    with archive.open(CSV_NAME) as file:
        df_preview = pd.read_csv(file, nrows=5)

print(f"Preview shape: {df_preview.shape}")

print("\nColumn names:")
print(df_preview.columns.tolist())

print("\nData types:")
print(df_preview.dtypes)

display(df_preview)

Preview shape: (5, 16)

Column names:
['index', 'Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover']

Data types:
index                         int64
Login Timestamp              object
User ID                       int64
Round-Trip Time [ms]        float64
IP Address                   object
Country                      object
Region                       object
City                         object
ASN                           int64
User Agent String            object
Browser Name and Version     object
OS Name and Version          object
Device Type                  object
Login Successful               bool
Is Attack IP                   bool
Is Account Takeover            bool
dtype: object


,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,0,2020-02-03 12:43:30.772,-4324475583306591935,NaN,10.0.65.171,NO,-,-,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Firefox 20.0.0.1618,iOS 13.4,mobile,False,False,False
1,1,2020-02-03 12:43:43.549,-4324475583306591935,NaN,194.87.207.6,AU,-,-,60117,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile 46.0.2490,Android 4.1,mobile,False,False,False
2,2,2020-02-03 12:43:55.873,-3284137479262433373,NaN,81.167.144.58,NO,Vestland,Urangsvag,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,3,2020-02-03 12:43:56.180,-4324475583306591935,NaN,170.39.78.152,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,False,False
4,4,2020-02-03 12:43:59.396,-4618854071942621186,NaN,10.0.0.47,US,Virginia,Ashburn,398986,Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA...,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False


### 3.7 Summary

The dataset has been successfully downloaded, verified, and inspected.

The initial preview contains 5 rows and 16 columns, including user identifiers, login timestamps, IP addresses, geographic information, device details, login outcomes, and security-related labels.

The dataset contains both numerical and categorical features, as well as Boolean variables.

The following observations were identified:

- `Login Timestamp` is initially stored as an object and will need datetime conversion.
- `User ID` is stored as an integer and can be used to identify user-related login patterns.
- `Login Successful`, `Is Attack IP`, and `Is Account Takeover` are Boolean variables.
- `Is Account Takeover` will be used as the evaluation target, not as an input feature during model training.
- `Is Attack IP` requires careful consideration to avoid potential data leakage.
- Some geographic fields contain placeholder values such as `-`, which will require further inspection during data preparation.

Since the dataset is large, we will use chunk-based processing in the next section to reduce memory consumption.

In the next section, we will prepare the dataset for unsupervised anomaly detection.



---



- Login Timestamp は初期状態では object 型として保存されているため、日時データ（datetime）への変換が必要です。

- User ID は整数型として保存されており、ユーザーに関連するログインパターンの特定に使用できます。

- Login Successful、Is Attack IP、Is Account Takeover はブール変数です。

- Is Account Takeover は評価ターゲットとして使用し、モデル訓練時の入力特徴量としては使用しません。

- Is Attack IP は、データリーク（情報の漏洩）の可能性を防ぐために慎重な検討が必要です。

- 一部の地理情報フィールドには - などのプレースホルダー値が含まれており、データ準備の段階でさらに確認が必要となります。

## 4 - Data Preparation

In this section, we prepare the dataset for unsupervised anomaly detection.

Since the original dataset contains millions of login events, we use chunk-based processing to reduce memory consumption.

We retain all Account Takeover (ATO) events and randomly sample a subset of non-ATO events for analysis.

The target label is kept separate from the input features to prevent data leakage.

### 4.1 Define Target and Feature Columns

We identify the target variable and select the initial features required for anomaly detection.

The `Is Account Takeover` column is reserved for model evaluation and will not be used during model training.

We initially select login-related and user-related information. Additional features will be engineered in a later section.

In [28]:
TARGET_COLUMN = "Is Account Takeover"

FEATURE_COLUMNS = [
    "User ID",
    "Login Timestamp",
    "Login Successful",
    "Country",
    "Device Type"
]

USE_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]

print("Target column:", TARGET_COLUMN)
print("\nSelected feature columns:")
print(FEATURE_COLUMNS)

Target column: Is Account Takeover

Selected feature columns:
['User ID', 'Login Timestamp', 'Login Successful', 'Country', 'Device Type']


### 4.2 Load and Sample the Dataset Using Chunk-based Random Sampling

To manage the large dataset efficiently, we use **Chunk-based Random Sampling** instead of loading the entire dataset into memory.

The CSV file is processed in chunks of 100,000 rows. Within each chunk, all positive Account Takeover (ATO) events are retained, while approximately 1% of negative events are randomly selected.

The sampled negative events from all chunks are then combined and limited to a maximum of 200,000 records.

This approach reduces memory consumption and computational overhead while obtaining randomly selected negative events from across the dataset.

The target label (`Is Account Takeover`) is used only to distinguish positive and negative events during sampling and for subsequent model evaluation. It is excluded from the input features used to train the unsupervised anomaly detection model.

Since this sampling strategy changes the original class distribution, evaluation metrics must be interpreted with this limitation in mind.

In [29]:
CHUNK_SIZE = 100_000
MAX_NEGATIVE_SAMPLES = 200_000

positive_chunks = []
negative_chunks = []

total_rows = 0

with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    with archive.open(CSV_NAME) as file:

        for chunk_number, chunk in enumerate(
            pd.read_csv(
                file,
                usecols=USE_COLUMNS,
                chunksize=CHUNK_SIZE
            )
        ):

            total_rows += len(chunk)

            # Keep all positive ATO events
            positive = chunk[chunk[TARGET_COLUMN] == True]

            if not positive.empty:
                positive_chunks.append(positive)

            # Randomly sample negative events from each chunk
            negative = chunk[chunk[TARGET_COLUMN] == False]

            sampled_negative = negative.sample(
                frac=0.01,
                random_state=RANDOM_STATE + chunk_number
            )

            if not sampled_negative.empty:
                negative_chunks.append(sampled_negative)

# Combine all positive events
positive_df = pd.concat(
    positive_chunks,
    ignore_index=True
)

# Combine sampled negative events
negative_df = pd.concat(
    negative_chunks,
    ignore_index=True
)

# Limit the negative sample to 200,000 rows
if len(negative_df) > MAX_NEGATIVE_SAMPLES:
    negative_df = negative_df.sample(
        n=MAX_NEGATIVE_SAMPLES,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

# Combine positive and negative events
df = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)

print("Dataset sampling completed.")
print(f"Total rows scanned: {total_rows:,}")
print(f"Positive ATO events: {len(positive_df):,}")
print(f"Negative events sampled: {len(negative_df):,}")
print(f"Final dataset shape: {df.shape}")

Dataset sampling completed.
Total rows scanned: 31,269,264
Positive ATO events: 141
Negative events sampled: 200,000
Final dataset shape: (200141, 6)


### 4.3 Validate the Sampled Dataset

After sampling, we validate the resulting dataset to ensure that it is suitable for further analysis.

The validation includes:

- Checking the dataset dimensions.
- Inspecting the distribution of the target variable.
- Checking for missing values.
- Checking for duplicate records.
- Reviewing the memory usage.

These checks help identify potential data quality issues before feature engineering and preprocessing.

In [30]:
# Check dataset dimensions
print("Dataset shape:", df.shape)

# Check target distribution
print("\nTarget distribution:")
print(df[TARGET_COLUMN].value_counts())

# Check target distribution in percentage
print("\nTarget distribution (%):")
print(
    (df[TARGET_COLUMN].value_counts(normalize=True) * 100)
    .round(4)
)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate records
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Check memory usage
memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemory usage: {memory_usage_mb:.2f} MB")

Dataset shape: (200141, 6)

Target distribution:
Is Account Takeover
False    200000
True        141
Name: count, dtype: int64

Target distribution (%):
Is Account Takeover
False    99.9295
True      0.0705
Name: proportion, dtype: float64

Missing values:
Login Timestamp        0
User ID                0
Country                0
Device Type            5
Login Successful       0
Is Account Takeover    0
dtype: int64

Number of duplicate rows: 0

Memory usage: 35.90 MB


#### 4.3.1 Interpretation
- Class imbalance: ATO events account for approximately 0.0705% of the total. This is higher than the original data's ~0.00045%, which is due to sampling the Negative events.

- Missing values: There are 5 missing values in the "Device Type" column.

- Duplicates: There are no duplicate rows.

- Memory: At approximately 35.90 MB, the sample size this time is relatively easy to work with.

### 4.4 Handle Missing Values and Convert Data Types

Before feature engineering, we handle missing values and ensure that each column has an appropriate data type.

The `Device Type` column contains a small number of missing values. These are replaced with `"Unknown"` to preserve the corresponding login events without introducing an unsupported device category.

The `Login Timestamp` column is converted from an object to a datetime format to support time-based feature engineering.

Finally, the dataset is checked again to verify that the changes were applied correctly.

In [31]:
# Replace missing device types
df["Device Type"] = df["Device Type"].fillna("unknown")

# Convert login timestamps to datetime
df["Login Timestamp"] = pd.to_datetime(
    df["Login Timestamp"],
    errors="coerce"
)

# Verify data types
print("Updated data types:")
print(df.dtypes)

# Check missing values again
print("\nMissing values after handling:")
print(df.isnull().sum())

# Check dataset dimensions
print("\nDataset shape:", df.shape)

Updated data types:
Login Timestamp        datetime64[ns]
User ID                         int64
Country                        object
Device Type                    object
Login Successful                 bool
Is Account Takeover              bool
dtype: object

Missing values after handling:
Login Timestamp        0
User ID                0
Country                0
Device Type            0
Login Successful       0
Is Account Takeover    0
dtype: int64

Dataset shape: (200141, 6)


#### 4.4.4 Interpretation

- Login Timestamp が datetime64[ns] に変換された。
- Device Type の欠損値5件が "unknown" に置き換えられた。
- すべての列で欠損値が0件になった。
- データ件数は変わらず200,141件。

In [32]:
df["Device Type"].unique()

array(['desktop', 'mobile', 'tablet', 'bot', 'unknown'], dtype=object)

### 4.5 Sort Events Chronologically

The sampled dataset contains positive and negative events that were collected separately and then combined.

As a result, the records are not necessarily in chronological order.

We sort the dataset by `Login Timestamp` to restore the chronological order of login events.

This step is important for subsequent time-based feature engineering and for preventing temporal data leakage during model evaluation.